# jevmark: RLCD from the SFT adapter, one arm and seed per session (task 2.2)

Thin wrapper: clone the private repo at one commit, install pinned dependencies, build the data, copy the SFT adapter of `SIZE` from the attached adapter dataset, run a smoke RLCD training, the 500-step run, then evaluate the result on all nine splits with the v1 protocol (`--shuffle-questions test_indomain`). All logic lives in the repo (`scripts/train_rlcd.py`). Setup, the adapter dataset and the stage 1 plan are in `docs/KAGGLE.md` section 10.

Settings: accelerator GPU T4 x2 (one is used), internet on, secret `GITHUB_TOKEN` attached, and the adapter dataset added as input (`ADAPTER_DATASET`).

In [ ]:
# Parameters: set before running. COMMIT must be a full 40-character sha.
REPO = "OWNER/jevmark"
COMMIT = "0000000000000000000000000000000000000000"
SIZE = "06b"          # "06b" or "17b"
ARM = "brier"         # outcome | outcome_minus_p | brier | log | direct_bandit | sft_cont
SEED = 0
ADAPTER_DATASET = "/kaggle/input/jevmark-sft-adapters"  # holds runs/sft_06b/adapter and runs/sft_17b/adapter (docs/KAGGLE.md section 10)
SMOKE_STEPS = 20      # optimizer steps in the smoke run
MAX_HOURS = 6.0       # training wall clock cap; leaves time for the evaluation


In [ ]:
# Clone REPO at COMMIT. The token reaches git only through environment variables,
# is never put on a command line or in .git/config, and is redacted from any output.
import base64
import os
import re
import subprocess
from pathlib import Path

from kaggle_secrets import UserSecretsClient

assert re.fullmatch(r"[\w.-]+/[\w.-]+", REPO), "REPO must be owner/name"
assert re.fullmatch(r"[0-9a-f]{40}", COMMIT), "COMMIT must be a full 40-character sha"
assert SIZE in ("06b", "17b"), "SIZE must be 06b or 17b"
assert ARM in ("outcome", "outcome_minus_p", "brier", "log", "direct_bandit", "sft_cont"), "unknown ARM"
# Let the CUDA caching allocator grow segments instead of fragmenting (decision 45). PyTorch 2.9 and later
# read PYTORCH_ALLOC_CONF; earlier releases read only PYTORCH_CUDA_ALLOC_CONF, so both are set. Every later
# !python inherits them.
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
WORK = Path("/tmp/jevmark")  # outside /kaggle/working, so the notebook output holds only runs/


def run(cmd, env=None, secrets=()):
    result = subprocess.run(cmd, cwd=WORK, env=env, capture_output=True, text=True)
    output = result.stdout + result.stderr
    for secret in secrets:
        output = output.replace(secret, "***")
    if output.strip():
        print(output[-4000:])
    if result.returncode != 0:
        raise RuntimeError(f"{cmd[0]} {cmd[1]} failed with exit code {result.returncode}")


token = UserSecretsClient().get_secret("GITHUB_TOKEN")
header = "AUTHORIZATION: basic " + base64.b64encode(f"x-access-token:{token}".encode()).decode()
git_env = {
    **os.environ,
    "GIT_TERMINAL_PROMPT": "0",
    "GIT_CONFIG_COUNT": "1",
    "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
    "GIT_CONFIG_VALUE_0": header,
}
WORK.mkdir(parents=True, exist_ok=True)
if not (WORK / ".git").exists():
    run(["git", "init", "-q"])
    run(["git", "remote", "add", "origin", f"https://github.com/{REPO}.git"])
run(["git", "fetch", "-q", "--depth", "1", "origin", COMMIT], env=git_env, secrets=(token, header))
run(["git", "checkout", "-q", "--force", "FETCH_HEAD"])
del token, header, git_env

head = subprocess.run(["git", "rev-parse", "HEAD"], cwd=WORK, capture_output=True, text=True, check=True).stdout.strip()
assert head == COMMIT, f"checked out {head}, expected {COMMIT}"
os.chdir(WORK)
print("checked out", head)

# Committed run directories are history, not state (decision 45): a fresh clone may hold the summary and log of an
# earlier run of this arm, which must never pass for this session's result. Delete them before training.
import shutil

RUN = f"rlcd_{SIZE}_{ARM}_s{SEED}"
for stale in (RUN, f"{RUN}_smoke"):
    shutil.rmtree(WORK / "runs" / stale, ignore_errors=True)
print("removed committed run directories:", f"{RUN}, {RUN}_smoke")


In [ ]:
# The Hugging Face packages pinned to uv.lock; Kaggle keeps its own torch and numpy (decision 23). jevmark itself without deps.
# Kaggle's preinstalled torchao 0.10 makes peft 0.21 raise on LoRA adapter injection; jevmark does not use it.
!pip uninstall -y -q torchao
!pip install -q -r requirements-kaggle.txt
!pip install -q -e . --no-deps
!python -c "import sys, torch, transformers, peft, datasets; print(sys.version.split()[0], 'torch', torch.__version__, 'cuda', torch.cuda.is_available(), torch.cuda.device_count(), 'transformers', transformers.__version__, 'peft', peft.__version__, 'datasets', datasets.__version__)"

In [ ]:
# Fail-fast (decision 45): a failing !python does not stop a cell, so every training cell checks its own result
# with jevmark.runcheck and raises on failure, and every evaluation cell refuses to run unless its training passed.
import datetime
import json
import shutil

from jevmark.runcheck import require_training

TRAINED = {}  # "fast", "smoke", "full" -> True once that training cell printed TRAINING PASS


def evaluation_allowed(stage):
    if not TRAINED.get(stage):
        raise RuntimeError(f"refusing to evaluate: the {stage} training cell did not print TRAINING PASS")


def check_exit(what, code):
    if code != 0:
        raise RuntimeError(f"{what} exited with code {code}")


In [ ]:
# Build the nine splits; the build fails loudly if any build check fails.
!make data-build PY=python


In [ ]:
# Copy the SFT adapter of SIZE from the adapter dataset into the clone: runs/sft_{SIZE}/adapter.
# train_rlcd.py records its sha256; compare it with the local runs/sft_{SIZE}/adapter (docs/KAGGLE.md section 10).
import hashlib

candidates = [Path(ADAPTER_DATASET) / "runs" / f"sft_{SIZE}" / "adapter", Path(ADAPTER_DATASET) / f"sft_{SIZE}" / "adapter"]
source = next((c for c in candidates if (c / "adapter_model.safetensors").is_file()), None)
if source is None:
    raise RuntimeError(f"no SFT adapter for {SIZE} under {ADAPTER_DATASET}; looked in {[str(c) for c in candidates]}")
target = WORK / "runs" / f"sft_{SIZE}" / "adapter"
shutil.rmtree(target, ignore_errors=True)
shutil.copytree(source, target)
print("SFT adapter", source, "->", target, "sha256", hashlib.sha256((target / "adapter_model.safetensors").read_bytes()).hexdigest())


In [ ]:
# Smoke run: SMOKE_STEPS steps into runs/{RUN}_smoke/ (gitignored), with the pre-flight memory check (policy and
# reference), a validation and the final valid passes. Prints TRAINING PASS or TRAINING FAIL and raises on FAIL.
started = datetime.datetime.now(datetime.timezone.utc)
!python scripts/train_rlcd.py --config configs/rlcd_{SIZE}.yaml --init runs/sft_{SIZE} arm={ARM} seed={SEED} run_name={RUN}_smoke --limit-steps {SMOKE_STEPS} --device cuda
require_training(WORK / "runs" / f"{RUN}_smoke", started, _exit_code)
TRAINED["smoke"] = True


In [ ]:
# The 500-step run into runs/{RUN}/. Prints TRAINING PASS or TRAINING FAIL and raises on FAIL, so no evaluation runs.
# If it stops at MAX_HOURS, runs/{RUN}/last holds the state for --resume.
if not TRAINED.get("smoke"):
    raise RuntimeError("refusing to train: the smoke run did not print TRAINING PASS")
started = datetime.datetime.now(datetime.timezone.utc)
!python scripts/train_rlcd.py --config configs/rlcd_{SIZE}.yaml --init runs/sft_{SIZE} arm={ARM} seed={SEED} --max-hours {MAX_HOURS} --device cuda
training_exit = _exit_code
shutil.copytree(WORK / "runs", "/kaggle/working/runs", dirs_exist_ok=True)  # keep the state even if the check fails
require_training(WORK / "runs" / RUN, started, training_exit)
TRAINED["full"] = True


In [ ]:
# Evaluate the best adapter on all nine splits with the v1 protocol (test_indomain also with its questions reordered):
# runs/{RUN}/metrics.json, results.jsonl.gz, plots/. Refuses to run unless the full training printed TRAINING PASS.
evaluation_allowed("full")
!python scripts/evaluate.py --ckpt runs/{RUN} --shuffle-questions test_indomain --device cuda
check_exit("evaluate.py", _exit_code)


In [ ]:
# Copy runs/ (including adapter/, adapter_last/ and last/) to /kaggle/working/runs for download; check only this session's run.
shutil.copytree(WORK / "runs", "/kaggle/working/runs", dirs_exist_ok=True)
metrics = json.loads((Path("/kaggle/working/runs") / RUN / "metrics.json").read_text())
assert metrics["git"]["commit"] == COMMIT, RUN
summary = json.loads((Path("/kaggle/working/runs") / RUN / "train_summary.json").read_text())
assert metrics["init_adapter_sha256"] == summary["init_adapter_sha256"], "evaluated run and training summary name different SFT adapters"
for split in ("test_indomain", "test_unseen_intents", "test_agnews", "test_emotion", "test_banking77", "test_yelp"):
    o = metrics["splits"][split]["overall"]
    print(f"{split:20} acc {o['accuracy']:.4f} ece {o['ece']:.4f} nll {o['nll']:.4f}")
print(RUN, "commit", metrics["git"]["commit"][:12], "dirty", metrics["git"]["dirty"], "fp32 fallback", metrics["precision"]["fp32_fallback_used"],
      "best step", summary["best_step"], f"{metrics['wall_clock_seconds'] / 60:.1f} min evaluation", f"{summary['wall_clock_seconds_this_session'] / 60:.1f} min training")
